# Seasonal Agriculture Performance Analysis
## VOIS AICTE Major Project

**Purpose:** Analyze seasonal differences in agricultural performance using data cleaning, descriptive statistics, visualization, correlation analysis and one-way ANOVA.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

df = pd.read_csv('../data/seasonal_agriculture_performance_dataset.csv')
df.shape


## 1. Dataset Understanding


In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nSeason counts:')
print(df['Season'].value_counts())
print('\nMissing values:')
print(df.isna().sum()[df.isna().sum() > 0])


## 2. Data Cleaning
Missing values in Rainfall, Soil Moisture and Yield are imputed with season-wise medians. Duplicate rows are removed.


In [ ]:
clean = df.copy()
for col in ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']:
    clean[col] = clean.groupby('Season')[col].transform(lambda s: s.fillna(s.median()))
clean = clean.drop_duplicates().reset_index(drop=True)
print('Cleaned shape:', clean.shape)
print('Remaining missing values:', clean.isna().sum().sum())


## 3. Seasonal Performance Summary


In [ ]:
season_order = ['Kharif', 'Rabi', 'Zaid']
summary = clean.groupby('Season').agg(
    Records=('Farm_ID','count'),
    Avg_Yield=('Yield_Tonnes_Ha','mean'),
    Avg_Revenue=('Revenue_INR','mean'),
    Avg_Cost=('Total_Cost_INR','mean'),
    Avg_Profit=('Profit_INR','mean'),
    Median_Profit=('Profit_INR','median'),
    Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3','mean'),
    Avg_Disease_Pest_Risk=('Disease_Pest_Risk_pct','mean')
).reindex(season_order)
summary.round(2)


## 4. Visualization


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(summary.index, summary['Avg_Yield'])
ax.set_title('Average Yield by Season')
ax.set_ylabel('Tonnes per hectare')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(summary.index, summary['Avg_Profit']/100000)
ax.set_title('Average Profit by Season')
ax.set_ylabel('₹ lakh')
plt.tight_layout(); plt.show()


## 5. Crop-Level Comparison


In [ ]:
crop_summary = clean.groupby('Crop')['Yield_Tonnes_Ha'].mean().sort_values(ascending=False)
crop_summary


In [ ]:
crop_summary.sort_values().plot(kind='barh', figsize=(8,5), title='Average Yield by Crop')
plt.xlabel('Tonnes per hectare'); plt.tight_layout(); plt.show()


## 6. Correlation Analysis
Correlation measures linear association. It does not establish causation.


In [ ]:
corr = clean.select_dtypes(include=np.number).corr()['Yield_Tonnes_Ha'].drop('Yield_Tonnes_Ha')
corr.sort_values(key=abs, ascending=False).head(10).round(3)


## 7. Statistical Validation — One-Way ANOVA


In [ ]:
profit_groups = [clean.loc[clean['Season']==s, 'Profit_INR'] for s in season_order]
yield_groups = [clean.loc[clean['Season']==s, 'Yield_Tonnes_Ha'] for s in season_order]
profit_test = f_oneway(*profit_groups)
yield_test = f_oneway(*yield_groups)
print('Profit ANOVA:', profit_test)
print('Yield ANOVA:', yield_test)


## 8. Conclusions
- Kharif has the highest average yield, revenue and profit in this dataset.
- Profit differences across seasons are statistically significant according to one-way ANOVA.
- Yield differences are not statistically significant at the 5% level in the ANOVA test.
- Water-efficiency shows the strongest observed linear association with yield among the numeric variables.
- Findings are dataset-specific and should support further investigation rather than universal conclusions.


## 9. Recommendations
1. Compare seasonal profitability alongside yield instead of using yield alone.
2. Investigate water-use efficiency as an important analytical factor associated with yield.
3. Examine crop- and region-level differences before making operational decisions.
4. Extend the study with multi-year data and stronger predictive models for future work.
